# Part 1: Knowledge bases and Microsoft Retriever 1

Cocoarynth Chocolate Works is a small bean-to-bar chocolate maker. Its records are scattered across farm logs, factory test results, shipping paperwork, retailer emails, and company policies. In this notebook, you'll build a knowledge base over those records and ask it a question.

**You'll:**
- Connect four search indexes as **knowledge sources** and combine them in a **knowledge base**.
- Ask a question, then check the answer's citations against the original document.
- Compare standard agentic retrieval with **Microsoft Retriever 1 (MR1)**, a faster retriever for when your app already knows which source to search.

Run the cells in order. Creating knowledge sources and knowledge bases and running retrieval calls use billable services.

## Step 1: Read the source document and load your settings

Before asking the knowledge base anything, open [the label and marketing claims policy](../../../data/cocoarynth/documents/cocoarynth-policies/label-claims-policy.pdf). Find the rule for when a bar may carry the organic seal. You'll compare the knowledge base's answer with this document later.

Then run the cell below to load your settings from the `.env` file.

**ℹ️ Note**
- The first time you run the cell below, you'll be prompted to select a kernel. Select **Python Environments** and choose the **.venv** environment.
- You may also be prompted with "Do you want to allow public and private networks to access this app?" Select **Allow**.

**⚠️ Troubleshooting:**
If code cells get stuck and keep spinning, try these remediation steps:
1. Select **Restart** from the notebook toolbar at the top. If that doesn't work:
2. Select **Reload window** from the VS Code command palette. If that doesn't work:
3. Close VS Code completely and restart it.

In [ ]:
import os
import sys
from pathlib import Path

from azure.core.credentials import AzureKeyCredential

REPO_ROOT = next(
    (directory for directory in (Path.cwd(), *Path.cwd().parents)
     if (directory / ".env.sample").is_file() and (directory / "src" / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from within the LAB673 repository.")
sys.path.insert(0, str(REPO_ROOT / "src" / "notebooks"))
from sdk_helpers import CLIENT_OPTIONS, SDK_VERSION, load_environment, required_env, service_endpoint

load_environment(REPO_ROOT)

AZURE_SEARCH_SERVICE_ENDPOINT = service_endpoint(required_env("AZURE_SEARCH_SERVICE_ENDPOINT"))
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)
print(f"Environment variables loaded; Search SDK: {SDK_VERSION}")

## Step 2: Check the indexes

Confirm that the four Cocoarynth indexes exist and contain documents. These are the same indexes you explored in the Azure portal. This notebook only reads them; it doesn't change them.

In [ ]:
from azure.search.documents import SearchClient

DOMAINS = ["origins", "quality", "logistics-retail", "policies"]
INDEX_NAMES = {domain: f"cocoarynth-{domain}" for domain in DOMAINS}

for index_name in INDEX_NAMES.values():
    with SearchClient(
        AZURE_SEARCH_SERVICE_ENDPOINT, index_name, search_credential, **CLIENT_OPTIONS
    ) as search_client:
        print(f"{index_name}: {search_client.get_document_count()} document chunks")

## Step 3: Create knowledge sources and a knowledge base

A **knowledge source** points to an existing index. A **knowledge base** combines one or more knowledge sources with the managed MR1 retriever. The standard configuration also connects a chat completion model for model-backed reasoning and answer synthesis.

The cell below creates four knowledge sources, one per index, and a knowledge base named `cocoarynth-kb-standard` that uses all four. If you run the cell again, it updates the objects you already created.

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

index_client = SearchIndexClient(
    AZURE_SEARCH_SERVICE_ENDPOINT, search_credential, **CLIENT_OPTIONS
)

KNOWLEDGE_SOURCE_NAMES = {domain: f"cocoarynth-{domain}-source" for domain in DOMAINS}
for domain, knowledge_source_name in KNOWLEDGE_SOURCE_NAMES.items():
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=f"Cocoarynth {domain} document passages",
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=INDEX_NAMES[domain],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            source_data_fields=[
                SearchIndexFieldReference(name=field)
                for field in ("chunk_id", "title", "chunk", "blob_path", "page_number_from", "page_number_to")
            ],
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source)
    print(f"Knowledge source '{knowledge_source_name}' created or updated.")

STANDARD_KB_NAME = "cocoarynth-kb-standard"
standard_knowledge_base = KnowledgeBase(
    name=STANDARD_KB_NAME,
    description="Standard agentic retrieval over the four Cocoarynth document sources",
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES.values()],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(
            azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                resource_url=AZURE_OPENAI_ENDPOINT,
                deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
                model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
                api_key=AZURE_OPENAI_KEY,
            )
        )
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    answer_instructions="Answer from the retrieved records, cite reference IDs, and say when evidence is missing.",
)
index_client.create_or_update_knowledge_base(standard_knowledge_base)
print(f"Knowledge base '{STANDARD_KB_NAME}' created or updated.")

## Step 4: Ask a question

Ask the knowledge base when a bar may carry the organic seal. The knowledge base plans one or more searches, retrieves passages from the relevant sources, and writes an answer with citations.

The output shows:
- The **answer**, with reference IDs.
- The **activity**: the planning, search, and answer synthesis steps the knowledge base took.
- The **references**: the passages it retrieved.

In [ ]:
import textwrap
import time

from IPython.display import Markdown, display

from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
)

QUESTION = "When may a Cocoarynth bar carry the organic seal, and what if the supplier certificate is pending renewal?"

standard_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=STANDARD_KB_NAME,
    credential=search_credential,
    **CLIENT_OPTIONS,
)
request = KnowledgeBaseRetrievalRequest(
    messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=QUESTION)])],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    include_activity=True,
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=name, include_references=True, include_reference_source_data=True
        )
        for name in KNOWLEDGE_SOURCE_NAMES.values()
    ],
)

start = time.perf_counter()
result = standard_client.retrieve(request)
print(f"Retrieval and answer synthesis took {(time.perf_counter() - start) * 1000:.0f} ms\n")
display(Markdown(result.response[0].content[0].text))

print("Activity:")
for activity in result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

### Check the answer against the source

- Find the policy reference in the output and read its passage.
- Reopen the original PDF. Does the answer correctly distinguish a certificate that's pending renewal from one that's valid on the packing date?
- Find the planning, search, and answer synthesis steps in the activity.

A citation shows which passage was retrieved. It doesn't prove that every sentence in the answer is correct, so it's worth checking.

## Step 5: Try Microsoft Retriever 1

Standard agentic retrieval uses a chat completion model to choose which sources to search and to plan queries. **Microsoft Retriever 1 (MR1)** is built for speed: it plans queries faster, but it doesn't choose sources. Your app tells it which source to search.

Here, we already know the question is about policy, so the cell below creates a second knowledge base, `cocoarynth-kb-mr1`, that searches only the policies source. It returns the matching passages rather than writing an answer.

In [ ]:
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalMinimalReasoningEffort

MR1_KB_NAME = "cocoarynth-kb-mr1"

mr1_knowledge_base = KnowledgeBase(
    name=MR1_KB_NAME,
    description="Policy-only Microsoft Retriever 1 evidence retrieval",
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE_NAMES["policies"])],
    models=[KnowledgeBaseMicrosoftRetriever1Model()],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(mr1_knowledge_base)
print(f"Knowledge base '{MR1_KB_NAME}' created or updated.")

## Step 6: Run the same question through MR1

To make a fair comparison, both knowledge bases search only the policies source and return passages, not a written answer. The standard knowledge base gets the question as a chat message with low reasoning effort. MR1 gets it as a search intent with minimal reasoning effort.

Run the cell and compare the returned passages with the ones from Step 4.

In [ ]:
from azure.search.documents.knowledgebases.models import KnowledgeRetrievalSemanticIntent

policies_only = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=KNOWLEDGE_SOURCE_NAMES["policies"],
    include_references=True,
    include_reference_source_data=True,
    always_query_source=True,
)


def standard_policy_request():
    other_sources = [
        SearchIndexKnowledgeSourceParams(knowledge_source_name=name, never_query_source=True)
        for domain, name in KNOWLEDGE_SOURCE_NAMES.items()
        if domain != "policies"
    ]
    return KnowledgeBaseRetrievalRequest(
        messages=[KnowledgeBaseMessage(role="user", content=[KnowledgeBaseMessageTextContent(text=QUESTION)])],
        retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[policies_only, *other_sources],
        max_output_documents=8,
    )


def mr1_request():
    return KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=QUESTION)],
        retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[policies_only],
        max_output_documents=8,
    )


mr1_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT,
    knowledge_base_name=MR1_KB_NAME,
    credential=search_credential,
    **CLIENT_OPTIONS,
)
mr1_result = mr1_client.retrieve(mr1_request())
print("Activity:")
for activity in mr1_result.activity:
    step = activity.as_dict()
    print(f"  {step['type']:<24} {step.get('knowledgeSourceName', ''):<32} results: {step.get('count', '-')}")

print("\nReferences:")
for reference in mr1_result.references:
    source_data = reference.source_data or {}
    print(f"  [{reference.id}] {source_data.get('title', reference.type)}")
    if "chunk" in source_data:
        print(textwrap.indent(textwrap.shorten(source_data["chunk"], 200), "      "))

## Step 7: Compare response times

The cell below sends the question to each knowledge base a few times and reports the median response time. It starts with one warm-up request per knowledge base that isn't counted.

These times include network latency and come from a small sample, so treat them as a rough demonstration rather than a benchmark. A faster response isn't necessarily a better one: compare the returned passages too.

In [ ]:
from statistics import median

TRIALS = 3

strategies = {
    "Standard (low effort)": (standard_client, standard_policy_request),
    "MR1 (minimal effort)": (mr1_client, mr1_request),
}
for client, make_request in strategies.values():
    client.retrieve(make_request())

for label, (client, make_request) in strategies.items():
    timings = []
    for _ in range(TRIALS):
        start = time.perf_counter()
        client.retrieve(make_request())
        timings.append((time.perf_counter() - start) * 1000)
    print(f"{label}: median {median(timings):.0f} ms (trials: {', '.join(f'{t:.0f}' for t in timings)})")

## Summary

- Use a **standard knowledge base** when the knowledge base needs to choose among several sources and write a grounded answer.
- Use **MR1** when your app already knows which source to search and speed matters most.

Your knowledge sources and knowledge bases stay in place for the next notebooks. Go back to the lab instructions to view them in the Azure portal, then continue to [Part 2: Entities and graphs](02-entities-and-graphs.ipynb).